In [1]:
import matminer
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, f1_score, precision_score, recall_score

In [2]:
df_sc_ef = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna()
df_sc_ep = pd.read_csv("df_sc_ep_ElementProperty_Magpie_ftd.csv").dropna()

In [3]:
df_sc_ef = df_sc_ef.iloc[:, 1:]
df_sc_ep = df_sc_ep.iloc[:, 1:].rename(columns={"Tc": "_Tc"})

In [4]:
print(df_sc_ef.shape)
print(df_sc_ep.shape)

(16375, 105)
(16375, 134)


In [5]:
merged_df = pd.merge(df_sc_ef, df_sc_ep, left_on=["_Tc", "_Composition"], right_on=["_Tc", "_Composition"], how="outer")
merged_df = merged_df[merged_df["_Tc"]>=10]
merged_df

,_Tc,_Composition,H,He,Li,Be,B,C,N,O,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.541925,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.571429,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,18.20,Nb3 Sn0.85 Tl0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.563319,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16377,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.536680,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16378,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.543379,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16385,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.571429,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16388,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.500000,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [6]:
X = merged_df.iloc[:, 2:]
y = merged_df['_Tc']

In [7]:
X

,H,He,Li,Be,B,C,N,O,F,Ne,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,2.110663,0.844265,1.013118,0.0,166.0,229.0,63.0,203.800000,30.240000,166.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.541925,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,106.949534,102.911141,12.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.571429,0.0,0.0,...,1.548471,0.013273,0.026318,0.0,12.0,225.0,213.0,95.447143,95.368163,12.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,141.0,229.0,88.0,208.987500,30.018750,229.0
11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.563319,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,101.290393,100.597910,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16377,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.536680,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,229.0,217.0,108.432432,103.506626,12.0
16378,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.543379,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,79.547293,91.032425,12.0
16385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.571429,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,95.391786,95.304898,12.0
16388,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.500000,0.0,0.0,...,0.000000,0.000000,0.000000,0.0,12.0,225.0,213.0,117.531250,105.531250,12.0


In [8]:
y

0        31.20
1        40.10
6        33.00
10       18.20
11       28.10
         ...  
16377    13.00
16378    22.00
16385    19.25
16388    63.60
16389    34.80
Name: _Tc, Length: 6250, dtype: float64

In [9]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [10]:
X_train.shape

(4687, 235)

In [11]:
# Build the model
model = RandomForestRegressor()

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 3, 5, 7],
    'min_samples_leaf': [1, 2, 3, 4],
    'bootstrap': [True, False]
}

In [ ]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Evaluate the model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 640 candidates, totalling 3200 fits


In [ ]:
# Print evaluation metrics
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))
print("R-squared:", r2_score(y_test, y_pred))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))
print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Adjusted R-squared:", r2_score(y_test, y_pred, multioutput='uniform_average'))

In [ ]:
pd.DataFrame(grid_search.cv_results_)

In [ ]:
filename = 'sc_efep_best_model_rf__3.pkl'
with open(filename, 'wb') as f:
    pickle.dump(best_model, f)

In [ ]:
# efep 700
# Mean Squared Error (MSE): 231.7748500192689
# R-squared: 0.6402716325478821
# Mean Absolute Error (MAE): 9.766499442563493
# Root Mean Squared Error (RMSE): 15.224153507478468
# Adjusted R-squared: 0.6402716325478821

# all

# Mean Squared Error (MSE): 107.35683492850448
# R-squared: 0.8445133016280957
# Mean Absolute Error (MAE): 5.156666694943521
# Root Mean Squared Error (RMSE): 10.361314343677856
# Adjusted R-squared: 0.8445133016280957

In [ ]:
# # Example: Create a new data point (features) for prediction
# new_data_point = [[feature1_value, feature2_value, ...]]

# # Predict using the loaded model
# predicted_value = loaded_model.predict(new_data_point)

# print("Predicted value:", predicted_value)


In [ ]:
# Mean Squared Error (MSE): 285.50798228513975
# R-squared: 0.5568746119199605
# Mean Absolute Error (MAE): 10.705040122111116
# Root Mean Squared Error (RMSE): 16.896981454838013
# Adjusted R-squared: 0.5568746119199605


# On 9000 rows
# Mean Squared Error (MSE): 138.87230308323188
# R-squared: 0.8021012250455234
# Mean Absolute Error (MAE): 6.131617541973627
# Root Mean Squared Error (RMSE): 11.784409322627583
# Adjusted R-squared: 0.8021012250455234

# Mean Squared Error (MSE): 127.29734257361541
# R-squared: 0.827537081963444
# Mean Absolute Error (MAE): 5.243760973322503
# Root Mean Squared Error (RMSE): 11.282612400220767
# Adjusted R-squared: 0.827537081963444